# JP2 to JPEG Conversion Notebook

# Overview 

This notebook is used to convert a large number of JPEG 2000 image files downloaded in the previous step to standard JPEG format using Python and ImageMagick. Converting the images to standard JPEG format makes the images compatible with common image viewers and web browsers. 

It searches the specified directory for JP2 files and converts them into the standard JPEG format. It converts multiple files at the same time using parallel processing.  

The converted JPEG files are stored in the same directory as the JPEG 2000 files, and the original JPEG 2000 files are simultaneously deleted. 

## Prerequisites

Before running the notebook to convert the JPEG 2000 files to JPEG, ensure you have the following:

- JP2 files extracted from the downloaded ZIP files.
- Sufficient storage space on your hard drive for the converted files.
- Python installed on your computer.
- Jupyter Notebook installed to open and run the code.
- ImageMagick installed, with the `mogrify` command available. This command is required to ensure the quality of the converted files are not corrupted.
- A backup of the original JP2 files if you want to retain them.

## Before Converting Files

Perform the following steps before converting the files:

### Install the Required Libraries

1. Open the `2-jp2-to-jpeg-conversion.ipynb` in the Jupyter interface.
2. Run the following code cell to import the libraries necessary for the conversion:

   ```python
   import os
   import time
   import subprocess
   from concurrent.futures import ThreadPoolExecutor
   import tqdm
   ```

Here's what each imported library does:

- `os`: Provides functions for interacting with the operating system
- `time`: Used for measuring the time taken for operations
- `subprocess`: Allows running external commands (like ImageMagick's 'mogrify')
- `ThreadPoolExecutor`: Enables parallel processing of tasks
- `tqdm`: Provides a progress bar for long-running operations

**Note:** The `tqdm` library is required only if the progress bar package is required. Installing the package shows a progress bar when the file conversion is in progress.

## Converting Files

### Specify the Directory Containing JPEG 2000 Files

Navigate to the following code snippet:

```python
jp2_files = []
for root, dirs, files in os.walk(input_dir):
    for file in files:
        if file.endswith('.jp2') and not file.startswith('._'):
            jp2_files.append(os.path.join(root, file))
```

Enter the name of the directory where the downloaded JPEG 2000 files are located in `input_dir = './downloads'`

**Note:** The code searches this folder and its subfolders for files ending in `.jp2`. Files beginning with `._` are excluded.

### Prepare the Conversion Function

Run the following code snippet:

```python
# Initialize variables
total_time = 0
file_count = len(jp2_files)
error_list = []  # List to record filenames of failed conversions

# Function to convert a single .jp2 file to .jpeg
def convert_and_remove(file_path):
    try:
        start_time = time.time()
        subprocess.run(['mogrify', '-format', 'jpeg', '-quality', '95', file_path], check=True)
        os.remove(file_path)  # Remove original file after successful conversion
        end_time = time.time()
        return end_time - start_time
    except Exception as e:
        error_list.append(file_path)  # Record the failing file path
        return 0  # Return zero time for failed conversions
```

This function initializes variables to keep track of the total time, file count, and any errors.

The `convert_and_remove` function uses ImageMagick to convert each JP2 file to JPEG with a quality setting of 95.

It then deletes the original JP2 file. If an error occurs, it records the affected file’s path.

### Start the Conversion of Files

Run the following code ThreadPoolExecutor snippet:

```python
# Use ThreadPoolExecutor to parallelize the file conversion
with ThreadPoolExecutor() as executor:
    # Optional: Use tqdm to show progress bar
    results = list(tqdm.tqdm(executor.map(convert_and_remove, jp2_files), total=file_count))

# Calculate total and average time
total_time = sum(results)
average_time = total_time / file_count if file_count > 0 else 0

# Output results and errors
print(f'Batch conversion complete.')
print(f'Total time: {total_time:.2f} seconds')
print(f'Average time per image: {average_time:.2f} seconds')
if error_list:
    print(f'Failed to convert {len(error_list)} files:')
    for error_file in error_list:
        print(error_file)
```

The `ThreadPoolExecutor` runs the conversion function on multiple files in parallel, which can significantly speed up the process on multi-core systems.

